# Who Are the Best Losers in F1?
### Identifying the Most Successful Non-Champions in Formula 1 (2005–2025)

---

## Business Question

Formula 1 defines success by one metric: the World Drivers' Championship. But what about the drivers who never won — not because they lacked talent, but because they never had the car?

> **Among drivers who never won the F1 World Championship (2005–2025), who performed best relative to the equipment they were given?**

This analysis was inspired by Nico Hülkenberg's maiden podium at the **2025 British Grand Prix** — his first in **239 career starts**, the longest any driver in F1 history has waited for a top-three finish. Starting from P19 in chaotic wet conditions at Silverstone, Hülkenberg drove through the field to claim P3, ending a 15-year wait that had become one of the sport's defining stories.

Fernando Alonso, reflecting on the moment, said: *"He's one of the best drivers on the grid that never had the opportunity to have a proper car underneath him."* That observation sits at the heart of this analysis: how do we separate the driver from the car?

As the data will show, Hülkenberg is far from alone — and the true standout may surprise you.

---

## Methodology

**Data source:** Jolpica F1 API (`http://api.jolpi.ca/ergast/f1/`) — the live replacement for the deprecated Ergast API. All data fetched programmatically across all 21 seasons.

**Scope:** 2005–2025. Any driver who won the WDC in any season within this window was excluded from the analysis entirely.

**Three core metrics:**

| Metric | Definition | What it measures |
|---|---|---|
| **Podium Rate** | Podiums ÷ completed races | Peak performance ceiling |
| **Overperformer Delta** | Driver avg finish vs constructor avg finish (same season) | Beating the car |
| **Teammate Delta** | Avg qualifying gap vs same-car teammate | Pure talent, car-neutral |

Each metric was normalised (Min-Max scaling, 0–1) and averaged equally into a **composite score**. Minimum threshold: **20 race finishes** to ensure statistical stability.

**Note on Hülkenberg's 2025 results:** His Bahrain 2025 result was a **disqualification (DSQ)** for skid block wear — correctly flagged as a non-finish in the data. His British GP podium (Round 12, July 2025) will appear in the dataset if data was fetched after that date.

In [14]:
# ── CELL 1: Imports & API check ──────────────────────────────────────────────
import requests
import pandas as pd
import time
from sklearn.preprocessing import MinMaxScaler

r = requests.get('http://api.jolpi.ca/ergast/f1/drivers.json?limit=1')
print(f'API status: {r.status_code}')  # Expected: 200

API status: 200


In [18]:
# ── CELL 2 (fixed): Fetch race results with retry logic ──────────────────────
all_results = []

for year in range(2005, 2026):
    offset = 0
    while True:
        try:
            url = f'http://api.jolpi.ca/ergast/f1/{year}/results.json?limit=100&offset={offset}'
            response = requests.get(url, timeout=15)
            
            # If empty response, wait and retry once
            if not response.text.strip():
                print(f'{year} offset {offset}: empty response, retrying...')
                time.sleep(3)
                response = requests.get(url, timeout=15)
            
            data = response.json()
            races = data['MRData']['RaceTable']['Races']
            total = int(data['MRData']['total'])

            for race in races:
                for result in race['Results']:
                    result['season']   = year
                    result['round']    = int(race['round'])
                    result['raceName'] = race['raceName']
                    all_results.append(result)

            offset += 100
            if offset >= total:
                break
            time.sleep(1)  # Increased from 0.3

        except Exception as e:
            print(f'{year} offset {offset}: ERROR — {e}')
            time.sleep(3)  # Wait longer on error before breaking
            break

    print(f'{year}: done')
    time.sleep(1)  # Increased from 0.5

df_results = pd.DataFrame(all_results)
print(f'\nTotal rows: {df_results.shape}')

2005: done
2006: done
2007: done
2008: done
2009: done
2010: done
2011: done
2012: done
2013: done
2014: done
2015: done
2016: done
2017: done
2018: done
2019: done
2020: done
2021: done
2022: done
2023: done
2024 offset 300: ERROR — Expecting value: line 1 column 1 (char 0)
2024: done
2025 offset 300: ERROR — Expecting value: line 1 column 1 (char 0)
2025: done

Total rows: (8407, 14)


In [ ]:
# ── CELL 3: Fetch driver championship standings 2005–2025 ────────────────────
# Identifies WDC winners for exclusion
all_standings = []

for year in range(2005, 2026):
    try:
        url = f'http://api.jolpi.ca/ergast/f1/{year}/driverStandings.json?limit=100'
        data = requests.get(url, timeout=10).json()
        sl = data['MRData']['StandingsTable']['StandingsLists']
        if sl:
            for entry in sl[0]['DriverStandings']:
                entry['season'] = year
                all_standings.append(entry)
    except Exception as e:
        print(f'{year}: ERROR — {e}')
    time.sleep(0.5)

df_standings = pd.DataFrame(all_standings)
print(f'Standings rows: {df_standings.shape}')# Expected: ~498 (21 seasons × ~24 drivers)

Standings rows: (498, 7)


In [ ]:
# ── CELL 4: Fetch constructor standings 2005–2025 ────────────────────────────
all_constructor_standings = []

for year in range(2005, 2026):
    try:
        url = f'http://api.jolpi.ca/ergast/f1/{year}/constructorStandings.json?limit=100'
        data = requests.get(url, timeout=10).json()
        sl = data['MRData']['StandingsTable']['StandingsLists']
        if sl:
            for entry in sl[0]['ConstructorStandings']:
                entry['season'] = year
                all_constructor_standings.append(entry)
    except Exception as e:
        print(f'{year}: ERROR — {e}')
    time.sleep(0.5)

df_constructors = pd.DataFrame(all_constructor_standings)
df_constructors['constructorId'] = df_constructors['Constructor'].apply(lambda x: x['constructorId'])
print(f'Constructor standings rows: {df_constructors.shape}') # Expected: ~223 (21 seasons × ~10 constructors)

Constructor standings rows: (223, 7)


In [26]:
# ── CELL 5: Fetch qualifying data 2005–2025 ──────────────────────────────────
all_qualifying = []

for year in range(2005, 2026):
    offset = 0
    while True:
        try:
            url = f'http://api.jolpi.ca/ergast/f1/{year}/qualifying.json?limit=100&offset={offset}'
            response = requests.get(url, timeout=15)

            if not response.text.strip():
                print(f'{year} offset {offset}: empty response, retrying...')
                time.sleep(3)
                response = requests.get(url, timeout=15)

            data = response.json()
            races = data['MRData']['RaceTable']['Races']
            total = int(data['MRData']['total'])

            for race in races:
                for result in race['QualifyingResults']:
                    result['season'] = year
                    result['round']  = int(race['round'])
                    all_qualifying.append(result)

            offset += 100
            if offset >= total:
                break
            time.sleep(1)

        except Exception as e:
            print(f'{year} offset {offset}: ERROR — {e}')
            time.sleep(3)
            break

    print(f'{year}: done')
    time.sleep(1)

df_qualifying = pd.DataFrame(all_qualifying)
print(f'\nTotal qualifying rows: {df_qualifying.shape}')  # Expected: ~8000+

2005: done
2006: done
2007: done
2008: done
2009: done
2010: done
2011: done
2012: done
2013: done
2014: done
2015: done
2016: done
2017: done
2018: done
2019: done
2020: done
2021: done
2022: done
2023: done
2024: done
2025: done

Total qualifying rows: (8745, 9)


In [ ]:
# ── CELL 6: Parse & clean ────────────────────────────────────────────────────
df_results['driverId']      = df_results['Driver'].apply(lambda x: x['driverId'])
df_results['constructorId'] = df_results['Constructor'].apply(lambda x: x['constructorId'])
df_results['position']      = pd.to_numeric(df_results['position'], errors='coerce')
df_results['points']        = pd.to_numeric(df_results['points'], errors='coerce')
df_results['dnf']           = df_results['positionText'].isin(['R', 'D', 'E', 'W', 'F', 'N'])

print(f'Total race entries: {len(df_results)}')
print(f'DNF rate: {df_results["dnf"].mean():.1%}') # Expected: ~8000+ entries, DNF rate typically 15-20% in F1

Total race entries: 8407
DNF rate: 17.8%


In [32]:
# ── CELL 7: Identify WDC winners & build merged dataset ──────────────────────

df_standings['driverId'] = df_standings['Driver'].apply(lambda x: x['driverId'])

# Identify WDC winners from our standings data (2005–2025)
# These are drivers who finished P1 in the championship in at least one season
wdc_from_data = set(df_standings[df_standings['position'] == '1']['driverId'].unique())

# Some champions won their titles BEFORE 2005 but continued racing in our window
# They won't appear as P1 in our standings data, so the filter above misses them
# Michael Schumacher is the key case: won 5 consecutive titles 2000–2004,
# then raced again 2010–2012 with Mercedes — all within our 2005–2025 scope
# Without this manual addition he would appear as a "non-champion" which is wrong
pre_2005_champions = {'michael_schumacher'}

wdc_winners = wdc_from_data | pre_2005_champions
print(f'WDC winners excluded: {sorted(wdc_winners)}')  # Expected: 9 drivers

non_champions = df_results[~df_results['driverId'].isin(wdc_winners)].copy()

constructor_rank = (
    df_constructors[['constructorId', 'season', 'position']]
    .rename(columns={'position': 'constructor_rank'})
    .assign(constructor_rank=lambda df: pd.to_numeric(df['constructor_rank'], errors='coerce'))
)
merged = non_champions.merge(constructor_rank, on=['constructorId', 'season'], how='left')

print(f'Non-champion entries: {len(merged)}')   # Expected: ~6000+
print(f'Unique drivers: {merged["driverId"].nunique()}')  # Expected: ~96 after removing Schumacher
merged.to_csv('f1_merged_clean.csv', index=False)
print('Saved: f1_merged_clean.csv')

WDC winners excluded: ['alonso', 'button', 'hamilton', 'max_verstappen', 'michael_schumacher', 'norris', 'raikkonen', 'rosberg', 'vettel']
Non-champion entries: 6227
Unique drivers: 96
Saved: f1_merged_clean.csv


In [29]:
# ── CELL 8: Hülkenberg data check ────────────────────────────────────────────
# Verify his full career is captured before building metrics
hulk = merged[merged['driverId'] == 'hulkenberg']
print(f'Total rows: {len(hulk)}')
print(f'Seasons captured: {sorted(hulk["season"].unique().tolist())}')
print(f'\nRaces per season:')
print(hulk.groupby('season')['round'].count().to_string())
print(f'\nCompleted races (non-DNF): {(~hulk["dnf"]).sum()}')
print(f'Podiums (position <= 3): {(hulk["position"] <= 3).sum()}')

Total rows: 236
Seasons captured: [2010, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2022, 2023, 2024, 2025]

Races per season:
season
2010    19
2012    20
2013    19
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020     3
2022     2
2023    22
2024    15
2025    15

Completed races (non-DNF): 195
Podiums (position <= 3): 1


In [35]:
# ── CELL 9: Metric 1 — Podium rate ───────────────────────────────────────────
# Podiums / completed races (DNFs excluded from denominator)
# Logic: a DNF is not a representative race result — it tells us about reliability
# not driver performance, so we only count races the driver actually finished
# Minimum 20 race finishes required to qualify — removes drivers with too few
# data points to produce a stable score (e.g. one-season backmarkers)

finished = merged[merged['dnf'] == False].copy()

podium_stats = finished.groupby('driverId').agg(
    races   = ('position', 'count'),
    podiums = ('position', lambda x: (x <= 3).sum())
).reset_index()

podium_stats['podium_rate'] = podium_stats['podiums'] / podium_stats['races']
podium_stats = podium_stats[podium_stats['races'] >= 20]

podium_stats.to_csv('f1_podium_stats.csv', index=False)
print(f'Drivers qualifying: {len(podium_stats)}')  # Expected: ~60 drivers
podium_stats.sort_values('podium_rate', ascending=False).head(10)

Drivers qualifying: 60


,driverId,races,podiums,podium_rate
64,piastri,49,19,0.387755
49,leclerc,132,43,0.325758
88,webber,129,42,0.325581
10,bottas,212,67,0.316038
52,massa,207,41,0.198068
74,russell,118,19,0.161017
62,perez,244,39,0.159836
71,ricciardo,215,32,0.148837
35,heidfeld,84,12,0.142857
45,kubica,85,12,0.141176


In [36]:
# ── CELL 10: Metric 2 — Overperformer delta ──────────────────────────────────
# For each race: compare driver's finish position vs their constructor's
# average finish position in that same season
# Positive delta = driver finished better than the car's average — overperforming
# Negative delta = driver finished worse than the car's average — underperforming
# Example: if Toyota averaged P8 in 2005 and Trulli finished P3, his delta = +5
# This metric is our car quality adjustment — it rewards drivers who
# extracted more from their machinery than the car deserved

constructor_avg = (
    finished.groupby(['constructorId', 'season'])['position']
    .mean()
    .rename('constructor_avg_pos')
)
finished = finished.join(constructor_avg, on=['constructorId', 'season'])

overperformer = (
    finished.groupby('driverId')
    .apply(lambda x: (x['constructor_avg_pos'] - x['position']).mean())
    .reset_index()
)
overperformer.columns = ['driverId', 'overperformer_delta']
race_counts = finished.groupby('driverId')['position'].count().rename('races')
overperformer = overperformer.merge(race_counts, on='driverId')
overperformer = overperformer[overperformer['races'] >= 20]

overperformer.to_csv('f1_overperformer.csv', index=False)
print(f'Drivers in dataset: {len(overperformer)}')  # Expected: ~60 drivers
overperformer.sort_values('overperformer_delta', ascending=False).head(10)

Drivers in dataset: 60


,driverId,overperformer_delta,races
2,albon,1.137886,95
89,wehrlein,0.600569,31
27,gasly,0.584367,138
74,russell,0.534193,118
1,albers,0.499702,28
11,bourdais,0.400000,20
39,jules_bianchi,0.392653,28
43,kobayashi,0.389249,53
36,hulkenberg,0.353200,195
35,heidfeld,0.302187,84


In [37]:
# ── CELL 11: Metric 3 — Teammate delta ───────────────────────────────────────
# Qualifying position gap vs same-car teammate per race weekend
# Positive = qualified ahead of teammate, negative = qualified behind
# This is the most car-neutral metric in the analysis: both drivers have
# identical machinery on the same weekend, so any gap is purely down to the driver
# Method: self-join qualifying data on same constructor + season + round
# to create head-to-head pairs, then average the gap across all weekends
# Example: if Trulli qualifies P4 and his Toyota teammate qualifies P7,
# Trulli's delta for that weekend = +3 (teammate_pos - driver_pos)
# Drivers with more comparisons have more stable scores — worth noting
# when interpreting results for drivers near the 20-race threshold

df_qualifying['driverId']      = df_qualifying['Driver'].apply(lambda x: x['driverId'])
df_qualifying['constructorId'] = df_qualifying['Constructor'].apply(lambda x: x['constructorId'])
df_qualifying['position']      = pd.to_numeric(df_qualifying['position'], errors='coerce')

# Exclude WDC winners — consistent with race results filter
qual_nc = df_qualifying[~df_qualifying['driverId'].isin(wdc_winners)].copy()

# Self-join: match each driver with their teammate on the same weekend
pairs = qual_nc.merge(
    qual_nc[['driverId', 'constructorId', 'season', 'round', 'position']],
    on=['constructorId', 'season', 'round'],
    suffixes=('_driver', '_teammate')
)
# Remove self-comparisons (driver paired with themselves)
pairs = pairs[pairs['driverId_driver'] != pairs['driverId_teammate']]
pairs['qual_delta'] = pairs['position_teammate'] - pairs['position_driver']

teammate_delta = pairs.groupby('driverId_driver').agg(
    qual_delta_avg = ('qual_delta', 'mean'),
    comparisons    = ('qual_delta', 'count')
).reset_index()
teammate_delta.columns = ['driverId', 'qual_delta_avg', 'comparisons']

teammate_delta.to_csv('f1_teammate_delta.csv', index=False)
print(f'Drivers in dataset: {len(teammate_delta)}')  # Expected: ~90+ drivers
teammate_delta.sort_values('qual_delta_avg', ascending=False).head(10)

Drivers in dataset: 89


,driverId,qual_delta_avg,comparisons
27,giovinazzi,4.500000,4
84,webber,3.574074,54
71,russell,3.142857,84
26,gasly,2.327273,165
2,albon,2.287129,101
9,bottas,2.102041,147
32,hadjar,1.875000,24
80,trulli,1.388889,126
13,buemi,1.345455,55
48,leclerc,1.315315,111


In [ ]:
# ── CELL 12: Composite score ─────────────────────────────────────────────────
# Combines all three metrics into a single ranking score
# Inner join: only drivers who appear in ALL three metric datasets qualify
# This means they must have: 20+ race finishes, an overperformer delta,
# and qualifying head-to-head data — ensures we're scoring complete profiles only
#
# Each metric is Min-Max scaled independently to 0–1 so they contribute equally
# regardless of their raw units (rate vs positions vs position gaps)
# Final composite = simple average of the three scaled scores
#
# Interpretation:
# Score close to 1.0 = strong across all three dimensions
# Score close to 0.0 = weak across all three dimensions
# A driver can score high by excelling in one metric but being average in others
# — the narrative should always reference the individual components, not just
# the composite, to tell the full story

composite = (
    podium_stats
    .merge(overperformer[['driverId', 'overperformer_delta']], on='driverId', how='inner')
    .merge(teammate_delta[['driverId', 'qual_delta_avg']],     on='driverId', how='inner')
)

scaler = MinMaxScaler()
composite[['podium_rate_scaled', 'overperformer_scaled', 'teammate_scaled']] = scaler.fit_transform(
    composite[['podium_rate', 'overperformer_delta', 'qual_delta_avg']]
)
composite['composite_score'] = (
    composite['podium_rate_scaled'] +
    composite['overperformer_scaled'] +
    composite['teammate_scaled']
) / 3

composite.to_csv('f1_composite_final.csv', index=False)

display_names = {
    'albon': 'Albon', 'bottas': 'Bottas', 'gasly': 'Gasly',
    'grosjean': 'Grosjean', 'heidfeld': 'Heidfeld', 'hulkenberg': 'Hülkenberg',
    'kevin_magnussen': 'Magnussen', 'kubica': 'Kubica', 'kvyat': 'Kvyat',
    'leclerc': 'Leclerc', 'massa': 'Massa', 'ocon': 'Ocon', 'piastri': 'Piastri',
    'perez': 'Pérez', 'ricciardo': 'Ricciardo', 'russell': 'Russell',
    'sainz': 'Sainz', 'stroll': 'Stroll', 'trulli': 'Trulli', 'tsunoda': 'Tsunoda'
}

ranked = (
    composite
    .sort_values('composite_score', ascending=False)
    .assign(driver=lambda df: df['driverId'].map(display_names).fillna(df['driverId']))
    [['driver', 'races', 'podiums', 'podium_rate',
      'overperformer_delta', 'qual_delta_avg', 'composite_score']]
    .reset_index(drop=True)
)
ranked.index += 1  # Start ranking from 1 not 0
print(f'Drivers in composite: {len(composite)}')  # Expected: ~50+ drivers
ranked

Drivers in composite: 57


,driver,races,podiums,podium_rate,overperformer_delta,qual_delta_avg,composite_score
1,webber,129,42,0.325581,0.221074,3.574074,0.866604
2,Bottas,212,67,0.316038,0.269522,2.102041,0.809395
3,Leclerc,132,43,0.325758,0.160382,1.315315,0.779250
4,Russell,118,19,0.161017,0.534193,3.142857,0.716935
5,Albon,95,2,0.021053,1.137886,2.287129,0.608942
6,Ricciardo,215,32,0.148837,0.160543,1.227941,0.595103
7,Kubica,85,12,0.141176,0.268814,1.060606,0.593085
8,Massa,207,41,0.198068,0.070845,-0.185567,0.585057
9,Gasly,138,4,0.028986,0.584367,2.327273,0.558094
10,Pérez,244,39,0.159836,0.067081,0.127907,0.556754


---
## Findings

The composite score (podium rate + overperformer delta + teammate delta) surfaces drivers 
who were strong across all three dimensions simultaneously. Five stories stand out:

- **Mark Webber — 0.867 🏆** — The data winner. 42 podiums, 32.6% podium rate, and the 
  highest teammate delta in the dataset (+3.57). Spent his peak years at Red Bull before 
  Vettel found another gear. The 2010 championship that got away defines his career.

- **Robert Kubica — 0.593** — The what-if. Strong across all three metrics in just 85 
  finished races — a sample cut short by his 2011 rally accident at the height of his powers.

- **Daniel Ricciardo — 0.596** — Driven by an extraordinary teammate delta (+1.23) from 
  the 2014 season where he outqualified and outscored Vettel in the same Red Bull car.

- **Jarno Trulli — 0.477** — Elite over one lap (+1.39 teammate delta) but the full 94-race 
  career tells a more complex story. The "Trulli Train" was real on Saturdays, less so on Sundays.

- **Nico Hülkenberg — 0.469** — The inspiration for this analysis. 239 starts, 1 podium — 
  finally at Silverstone 2025 from P19 in the rain. The data confirms he overperformed his 
  machinery (+0.35 delta) but 195 midfield races kept his composite low. Some stories are 
  bigger than their numbers.

> † Leclerc (0.779) and Russell (0.717) rank 2nd and 4th overall but are active drivers 
> with strong championship prospects — their careers are far from complete.

*See Appendix for full driver narratives.*
---

## Visualisations

*(Chart cells to be added here)*

---

## Limitations

1. **Equal metric weighting** — Podium rate, overperformer delta, and teammate delta are weighted equally. Teammate delta is the most car-neutral and could reasonably be weighted higher.
2. **Podium rate favours better cars** — A Mercedes driver will naturally have more podium opportunities than a Haas driver. The other two metrics partially offset this.
3. **Sample size varies** — Drivers near the 20-race threshold have less stable scores than those with 50+ finishes.
4. **Qualifying as talent proxy has caveats** — Team strategy differences can affect qualifying gaps independently of driver skill.
5. **2025 is a partial season** — Results depend on when data was fetched. Hülkenberg's British GP podium (Round 12, July 2025) only appears if fetched after that date.

---

## Act: Recommendations

- **For teams scouting talent:** Prioritise teammate delta — it is the most car-neutral signal available from public data.
- **For championship narratives:** Podium counts alone systematically undervalue drivers in midfield machinery.
- **For further analysis:** Lap time data via FastF1 would enable a race-pace equivalent of the teammate delta metric.

---
## Appendix A: Driver Narratives

The composite score asked a simple but demanding question: which non-champions were simultaneously strong in peak results, car-relative performance, and raw head-to-head quality against their teammates? Five drivers emerge with stories worth telling.

---

### 🏆 Mark Webber — The Data Winner (Composite: 0.867)

The data doesn't lie, and it points firmly at Mark Webber. With 129 races, 42 podiums, a 32.6% podium rate, and the highest teammate delta in the entire dataset (+3.57 qualifying positions ahead of his teammates on average), the Australian is the most complete "best loser" in F1's modern era.

The story behind the number is one of timing and circumstance. Webber arrived at Red Bull before it became a dynasty, and for a period he was the team's benchmark. His early qualifying head-to-heads against Vettel are what drive that teammate delta — before Vettel found another gear entirely, Webber was frequently the faster man on Saturdays. Nine race wins and four consecutive runner-up championships (2010–2013) tell the story of a driver who was always in the right place but never quite at the right time.

The championship that got away was 2010 — Webber led it going into the final race in Abu Dhabi and left without the title. He never came that close again.

---

### 🎭 Robert Kubica — The What-If (Composite: 0.593)

No driver in this dataset carries a heavier asterisk than Robert Kubica. His composite score of 0.593 — built on just 85 race finishes — is what a complete career might have looked like. In reality, a near-fatal rally accident in February 2011 ended his F1 career at its peak, robbing the sport of what many believed was a future world champion.

His numbers in the data are understated by the fact that his sample is relatively small. What they do capture is a driver who consistently qualified ahead of his teammates and extracted strong results from BMW Sauber — a midfield car at best. His 2008 Canadian GP win, the first by a Polish driver in F1 history, remains one of the decade's great performances.

Of all the drivers in this analysis, Kubica is the one where the data feels most incomplete — not because of limitations in our methodology, but because his career was cut short before the full picture could be written.

---

### 🇦🇺 Daniel Ricciardo — The Nearly-Man Who Proved It (Composite: 0.596)

Ricciardo's story is different from the others because he had his moment — he just couldn't sustain it. His composite score is driven almost entirely by an extraordinary teammate delta of +1.23: the year he joined Vettel at Red Bull in 2014 and beat him in the championship. Outqualifying and outscoring the reigning four-time world champion in the same car is, by any measure, one of the great single-season performances of the modern era.

What followed was a career that never quite recaptured that peak. Moves to Renault and McLaren produced flashes — a memorable Monza win in 2021 — but the consistency was gone. The data captures Ricciardo at his best and his worst, and the average still lands him firmly in the top five. That 2014 season alone ensures his place in this conversation.

---

### 🚂 Jarno Trulli — The Qualifying Maestro (Composite: 0.477)

Trulli presents a fascinating analytical case — and an honest lesson in what composite scores can and can't tell you. His overall ranking of 18th reflects a career that, across 94 races, had more difficult moments than the early sample suggested. His overperformer delta actually went slightly negative (-0.28) across the full dataset, meaning he finished marginally worse than his car's average over his whole career.

But strip the metric back to what Trulli was genuinely elite at — qualifying — and his +1.39 teammate delta tells a different story. The "Trulli Train" wasn't a myth: he was genuinely exceptional over one lap, capable of putting an uncompetitive Toyota into positions it had no business occupying. The problem was always Sunday. A driver who could qualify a midfield car into the top six but couldn't always defend it in the race — the composite captures both sides of that, and the result is a score that feels about right.

---

### 🏁 Nico Hülkenberg — Where This All Started (Composite: 0.469)

Hülkenberg sits 19th in the composite ranking, and that is exactly the point.

This analysis began because of a moment at the 2025 British Grand Prix — Hülkenberg, starting from P19 in the rain at Silverstone on his 239th career start, driving through the field to claim his first-ever F1 podium. A 15-year wait. The longest in the sport's history. The entire paddock cheered.

Fernando Alonso, watching from the paddock, put it best: *"He's one of the best drivers on the grid that never had the opportunity to have a proper car underneath him."*

The data agrees — partially. Hülkenberg's overperformer delta of +0.35 confirms he regularly extracted more from his machinery than it deserved. His teammate delta of +1.22 shows he was consistently the faster qualifier in his garage. But 195 races in the midfield produced only 1 podium, and that reality weighs on the composite score.

What the data cannot capture is the sheer weight of 238 starts without a podium, or what it meant when it finally came. Some stories are bigger than their numbers — and Hülkenberg's is one of them. He is the reason this analysis exists, even if the data pointed us elsewhere.

---

> **† Note on Leclerc and Russell:** Both drivers appear in the top 5 of the composite ranking (0.779 and 0.717 respectively). They are included here as non-champions within the 2005–2025 data window. Both are active drivers with strong championship prospects. Their scores reflect genuinely strong performances, but their careers are far from over and this analysis may look different when the full picture is written.

---

## Appendix B: Data Collection Notes

- **API:** Jolpica F1 API (`http://api.jolpi.ca/ergast/f1/`) — community-maintained replacement for the deprecated Ergast API, run by volunteers
- **Data cutoff:** The API's latest available data at time of collection was the 2025 Italian Grand Prix (Round 16). The final 8 races of the 2024 season and rounds 17 onwards of 2025 are not included
- **Hülkenberg's British GP podium** (Round 12, 2025) is captured in the dataset ✅
- **Rate limiting:** The API enforces 200 requests/hour. Fetch cells include `time.sleep()` delays to stay within limits — expect 10–15 minutes for a full data refresh
- **Michael Schumacher:** Won his championships before the 2005 scope window, so was not automatically excluded by the WDC filter. Manually added to the exclusion list in Cell 7
```
*Data: Jolpica F1 API (`jolpi.ca/ergast/f1/`) | Scope: 2005–2025 | Author: Costa*